In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Aapki uploaded file 'drug200.csv' ko direct read karega
data= pd.read_csv('drug200.csv')

In [11]:
data

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY
...,...,...,...,...,...,...
195,56,F,LOW,HIGH,11.567,drugC
196,16,M,LOW,HIGH,12.006,drugC
197,52,M,NORMAL,HIGH,9.894,drugX
198,23,M,NORMAL,NORMAL,14.020,drugX


In [12]:
print(f"Dataset Shape: {df.shape}\n")
print("--- Missing Values ---")
print(data.isnull().sum(), "\n")
print("--- Target Distribution ---")
print(data['Drug'].value_counts(), "\n")

# Head output Jupyter mein automatic professional column-wise matrix display karega
data.head()

Dataset Shape: (200, 6)

--- Missing Values ---
Age            0
Sex            0
BP             0
Cholesterol    0
Na_to_K        0
Drug           0
dtype: int64 

--- Target Distribution ---
Drug
drugY    91
drugX    54
drugA    23
drugC    16
drugB    16
Name: count, dtype: int64 



,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY


In [13]:
data_trad = data.copy()

# Encodings
data_trad['Sex'] = LabelEncoder().fit_transform(data_trad['Sex'])
data_trad['BP'] = data_trad['BP'].map({'LOW': 0, 'NORMAL': 1, 'HIGH': 2})
data_trad['Cholesterol'] = data_trad['Cholesterol'].map({'NORMAL': 0, 'HIGH': 1})

# Features (X) aur Target (y)
X_trad = data_trad[['Age', 'Sex', 'BP', 'Cholesterol', 'Na_to_K']]
y_trad = data_trad['Drug']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_trad, y_trad, test_size=0.20, random_state=42)

In [14]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Support Vector Machine": SVC(random_state=42)
}

accuracy_results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy_results[name] = accuracy_score(y_test, y_pred)
    
    print(f"=== {name} ===")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred), "\n")

=== Logistic Regression ===
Confusion Matrix:
 [[ 6  0  0  0  0]
 [ 0  3  0  0  0]
 [ 0  0  1  4  0]
 [ 0  0  0 11  0]
 [ 0  0  0  0 15]]
Classification Report:
               precision    recall  f1-score   support

       drugA       1.00      1.00      1.00         6
       drugB       1.00      1.00      1.00         3
       drugC       1.00      0.20      0.33         5
       drugX       0.73      1.00      0.85        11
       drugY       1.00      1.00      1.00        15

    accuracy                           0.90        40
   macro avg       0.95      0.84      0.84        40
weighted avg       0.93      0.90      0.87        40
 

=== Decision Tree ===
Confusion Matrix:
 [[ 6  0  0  0  0]
 [ 0  3  0  0  0]
 [ 0  0  5  0  0]
 [ 0  0  0 11  0]
 [ 0  0  0  0 15]]
Classification Report:
               precision    recall  f1-score   support

       drugA       1.00      1.00      1.00         6
       drugB       1.00      1.00      1.00         3
       drugC       1.00     

In [15]:
comparison_data = pd.DataFrame(list(accuracy_results.items()), columns=['Model Name', 'Accuracy'])
comparison_data

,Model Name,Accuracy
0,Logistic Regression,0.900
1,Decision Tree,1.000
2,Random Forest,0.975
3,K-Nearest Neighbors,0.700
4,Support Vector Machine,0.625


In [8]:
# 5 Custom mareezon ka data neat columns mein
custom_patients = pd.DataFrame({
    'Age': [23, 47, 65, 34, 55],
    'Sex': [0, 1, 1, 0, 1],                
    'BP': [2, 0, 1, 2, 1],                 
    'Cholesterol': [1, 1, 0, 0, 1],        
    'Na_to_K': [25.3, 11.2, 14.5, 31.8, 8.4]
})

# Sabiha sabse behtreen model uthayen (Decision Tree)
best_model_name = comparison_data.loc[comparison_data['Accuracy'].idxmax()]['Model Name']
best_model = models[best_model_name]

# Predictions ko naye column 'Prescribed_Drug' mein insert karna
custom_patients['Prescribed_Drug'] = best_model.predict(custom_patients)

# Grid style display ke liye direct cell variable return
custom_patients

,Age,Sex,BP,Cholesterol,Na_to_K,Prescribed_Drug
0,23,0,2,1,25.3,drugY
1,47,1,0,1,11.2,drugC
2,65,1,1,0,14.5,drugX
3,34,0,2,0,31.8,drugY
4,55,1,1,1,8.4,drugX


In [16]:
X_pipe = data[['Age', 'Sex', 'BP', 'Cholesterol', 'Na_to_K']]
y_pipe = data['Drug']

# ColumnTransformer scaling aur encoding ke liye
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Age', 'Na_to_K']),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ['Sex', 'BP', 'Cholesterol'])
    ]
)

# Automated Pipeline
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_pipe, y_pipe, test_size=0.20, random_state=42)
pipeline_rf.fit(X_train_p, y_train_p)

pipe_acc = accuracy_score(y_test_p, pipeline_rf.predict(X_test_p))
print(f"Pipeline Random Forest Accuracy: {pipe_acc:.4f}")

Pipeline Random Forest Accuracy: 1.0000
